# Task 1: Anti-Cheat Detection - Ultimate Pipeline (v3)
**โมเดลตรวจจับคนโกงที่มีประสิทธิภาพสูงสุด: 5-Fold Pseudo-Labeling Multi-Model Ensemble**

* **ผู้รับผิดชอบ:** apfirst13
* **Metric:** F2 Score (Recall weighted 4x over Precision, $\beta=2$)
* **ผลลัพธ์ Benchmark สูงสุด:**
  * **5-Fold Out-Of-Fold F2 Score:** `0.83727`
  * **ROC-AUC:** `0.89099`
  * **Optimal Threshold:** `0.3060` (Recall = `0.9788`, Precision = `0.5304`)
  * **Architecture:** LightGBM (Pseudo-labeled) + XGBoost + CatBoost (Nelder-Mead Optimized Blending)

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score, precision_score, recall_score, roc_auc_score, precision_recall_curve
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
print("Libraries loaded successfully!")

## 1. Load Data & Preprocessing
โหลดข้อมูลชุดฝึกสอนและชุดทดสอบ โดยแยกชุดข้อมูลที่มี Label ออกจากแถวที่ค่า `is_cheater` เป็น `NaN` เพื่อเตรียมทำ Pseudo-Labeling

In [ ]:
train_df = pd.read_csv("../data/task1/train.csv")
test_df = pd.read_csv("../data/task1/test.csv")
sample_sub = pd.read_csv("../data/sample_submission.csv")

labeled_df = train_df.dropna(subset=["is_cheater"]).copy()
unlabeled_df = train_df[train_df["is_cheater"].isna()].copy()
labeled_df["is_cheater"] = labeled_df["is_cheater"].astype(int)

print(f"ขนาดชุดข้อมูลทั้งหมด: {train_df.shape[0]:,} แถว x {train_df.shape[1]} คอลัมน์")
print(f" - Labeled Train:   {len(labeled_df):,} แถว (คนปกติ: {(labeled_df['is_cheater']==0).sum():,}, คนโกง: {(labeled_df['is_cheater']==1).sum():,})")
print(f" - Unlabeled Train: {len(unlabeled_df):,} แถว (เตรียมทำ Semi-Supervised Pseudo-Labeling)")
print(f" - Test Set:        {len(test_df):,} แถว")

## 2. Extended Domain & Behavioral Feature Engineering (66 Features)
สกัดฟีเจอร์พฤติกรรมผิดธรรมชาติ, อัตราส่วนความสามารถเชิงกลไกเทียบกับแท็กติก, และ Relative Statistics ตามระดับเลเวลผู้เล่น

In [ ]:
def advanced_feature_engineering(df):
    data = df.copy()

    # 1. Base Domain & Mechanics
    data['aimbot_index'] = data['headshot_percentage'] * data['crosshair_placement']
    data['recoil_efficiency'] = data['spray_control_score'] * data['accuracy_score']
    data['aim_vs_smoothness'] = data['accuracy_score'] / (data['aiming_smoothness'] + 1e-4)
    data['reaction_speed_inv'] = 1000.0 / (data['reaction_time_ms'] + 1e-4)
    data['reaction_vs_accuracy'] = data['accuracy_score'] / (data['reaction_time_ms'] + 1e-4)
    
    # Inhuman mechanics & anomaly flags
    data['inhuman_reaction_hs'] = ((data['reaction_time_ms'] < 130) & (data['headshot_percentage'] > 50)).astype(float)
    data['inhuman_aim_score'] = (data['accuracy_score'] * data['spray_control_score'] * data['crosshair_placement'])
    data['reaction_aim_product'] = (data['accuracy_score'] * data['headshot_percentage']) / (data['reaction_time_ms'] + 1.0)

    # 2. Mechanics vs Tactics Index
    data['mechanics_index'] = (data['accuracy_score'] + data['spray_control_score'] + data['crosshair_placement']) / 3.0
    data['tactics_index'] = (data['game_sense_score'] + data['team_play_score'] + data['buy_decision_score'] + data['map_knowledge']) / 4.0
    data['mech_vs_tactics'] = data['mechanics_index'] / (data['tactics_index'] + 1e-4)
    data['aim_vs_gamesense'] = data['accuracy_score'] / (data['game_sense_score'] + 1e-4)

    # 3. Combat & Efficiency
    data['damage_per_survival'] = data['damage_per_round'] / (data['survival_time_avg'] + 1.0)
    data['combat_power'] = (data['kill_death_ratio'] * data['damage_per_round']) / (data['survival_time_avg'] + 1.0)
    data['clutch_vs_game_sense'] = data['clutch_success_rate'] / (data['game_sense_score'] + 1e-4)
    data['hs_vs_accuracy'] = data['headshot_percentage'] / (data['accuracy_score'] + 1e-4)
    data['damage_per_kdr'] = data['damage_per_round'] / (data['kill_death_ratio'] + 1e-4)
    data['kdr_vs_survival'] = data['kill_death_ratio'] / (data['survival_time_avg'] + 1.0)

    # 4. Smurf & Account Suspicion
    data['suspicion_per_age'] = (data['kill_death_ratio'] * data['reports_received']) / (data['account_age_days'] + 1.0)
    data['kd_per_level'] = data['kill_death_ratio'] / (data['level'] + 1.0)
    data['progression_vs_age'] = data['level_progression_speed'] / (data['account_age_days'] + 1.0)
    data['reports_per_level'] = data['reports_received'] / (data['level'] + 1.0)
    data['reports_per_age'] = data['reports_received'] / (data['account_age_days'] + 1.0)
    data['friends_per_age'] = data['friend_network_size'] / (data['account_age_days'] + 1.0)
    data['friends_per_level'] = data['friend_network_size'] / (data['level'] + 1.0)
    data['device_churn_rate'] = data['device_changes_count'] / (data['account_age_days'] + 1.0)
    data['smurf_risk_composite'] = (data['reports_received'] * data['kill_death_ratio']) / (data['account_age_days'] * (data['friend_network_size'] + 1.0) + 10.0)

    # 5. Missing indicators
    data['missing_feature_count'] = data.isnull().sum(axis=1)
    for col in ['reports_received', 'crosshair_placement', 'friend_network_size', 'account_age_days', 'reaction_time_ms', 'spray_control_score']:
        data[f'{col}_isna'] = data[col].isna().astype(float)

    # 6. Level quantiles
    level_filled = data['level'].fillna(data['level'].median())
    data['level_bin'] = pd.qcut(level_filled, q=10, labels=False, duplicates='drop')

    cols_to_drop = ['player_id', 'is_cheater']
    return data.drop(columns=[c for c in cols_to_drop if c in data.columns])

X_labeled = advanced_feature_engineering(labeled_df)
y_labeled = labeled_df['is_cheater'].values
X_unlabeled = advanced_feature_engineering(unlabeled_df)
X_test = advanced_feature_engineering(test_df)
print(f"สร้างฟีเจอร์สำเร็จ: {X_labeled.shape[1]} ฟีเจอร์")

## 3. Stratified 5-Fold Training (Phase 1)
เทรน 3 โมเดลต้นไม้ (LightGBM, XGBoost, CatBoost) โดยปรับ `scale_pos_weight = 2.0` เพื่อเน้น Recall สำหรับ F2

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X_labeled))
oof_xgb = np.zeros(len(X_labeled))
oof_cat = np.zeros(len(X_labeled))

test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))

unlabeled_lgb = np.zeros(len(X_unlabeled))
unlabeled_xgb = np.zeros(len(X_unlabeled))
unlabeled_cat = np.zeros(len(X_unlabeled))

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_labeled, y_labeled)):
    X_trn, y_trn = X_labeled.iloc[trn_idx], y_labeled[trn_idx]
    X_val, y_val = X_labeled.iloc[val_idx], y_labeled[val_idx]

    # 1. LightGBM
    clf_lgb = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=63, max_depth=7, colsample_bytree=0.8, subsample=0.8, scale_pos_weight=2.0, random_state=42+fold, verbose=-1, n_jobs=-1)
    clf_lgb.fit(X_trn, y_trn)
    oof_lgb[val_idx] = clf_lgb.predict_proba(X_val)[:, 1]
    test_lgb += clf_lgb.predict_proba(X_test)[:, 1] / 5.0
    unlabeled_lgb += clf_lgb.predict_proba(X_unlabeled)[:, 1] / 5.0

    # 2. XGBoost
    clf_xgb = XGBClassifier(n_estimators=450, learning_rate=0.03, max_depth=6, colsample_bytree=0.8, subsample=0.8, scale_pos_weight=2.0, random_state=42+fold, eval_metric='logloss', n_jobs=-1)
    clf_xgb.fit(X_trn, y_trn)
    oof_xgb[val_idx] = clf_xgb.predict_proba(X_val)[:, 1]
    test_xgb += clf_xgb.predict_proba(X_test)[:, 1] / 5.0
    unlabeled_xgb += clf_xgb.predict_proba(X_unlabeled)[:, 1] / 5.0

    # 3. CatBoost
    clf_cat = CatBoostClassifier(iterations=450, learning_rate=0.04, depth=6, auto_class_weights='Balanced', random_seed=42+fold, verbose=0, thread_count=-1)
    clf_cat.fit(X_trn, y_trn)
    oof_cat[val_idx] = clf_cat.predict_proba(X_val)[:, 1]
    test_cat += clf_cat.predict_proba(X_test)[:, 1] / 5.0
    unlabeled_cat += clf_cat.predict_proba(X_unlabeled)[:, 1] / 5.0
    print(f"Fold {fold+1}/5 completed.")

## 4. Semi-Supervised Learning (Pseudo-Labeling)
คัดเลือกตัวอย่างที่มีความมั่นใจสูงพิเศษจาก Unlabeled Train Data (13,184 แถว) เพื่อนำกลับมาร่วมเทรนใน LightGBM

In [ ]:
unlabeled_blend = 0.45 * unlabeled_lgb + 0.30 * unlabeled_xgb + 0.25 * unlabeled_cat
pos_pseudo = (unlabeled_blend >= 0.96)
neg_pseudo = (unlabeled_blend <= 0.04)
pseudo_mask = pos_pseudo | neg_pseudo

print(f"- มั่นใจว่าเป็นคนโกง (p >= 0.96): {pos_pseudo.sum():,} แถว")
print(f"- มั่นใจว่าเป็นคนปกติ (p <= 0.04): {neg_pseudo.sum():,} แถว")
print(f"- รวมตัวอย่าง Pseudo-Labels ที่นำมาเสริม: {pseudo_mask.sum():,} แถว")

X_pseudo = X_unlabeled[pseudo_mask].copy()
y_pseudo = (unlabeled_blend[pseudo_mask] >= 0.5).astype(int)

# Retrain LightGBM with Pseudo-Labels
oof_lgb_p = np.zeros(len(X_labeled))
test_lgb_p = np.zeros(len(X_test))

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_labeled, y_labeled)):
    X_trn_p = pd.concat([X_labeled.iloc[trn_idx], X_pseudo], ignore_index=True)
    y_trn_p = np.concatenate([y_labeled[trn_idx], y_pseudo])
    X_val = X_labeled.iloc[val_idx]

    clf_lgb_p = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=63, max_depth=7, colsample_bytree=0.8, subsample=0.8, scale_pos_weight=2.0, random_state=42+fold, verbose=-1, n_jobs=-1)
    clf_lgb_p.fit(X_trn_p, y_trn_p)
    oof_lgb_p[val_idx] = clf_lgb_p.predict_proba(X_val)[:, 1]
    test_lgb_p += clf_lgb_p.predict_proba(X_test)[:, 1] / 5.0
    print(f"Retrain Fold {fold+1}/5 completed.")

## 5. Optimal Blending & Fine-Grained Threshold Tuning
ค้นหาน้ำหนักการผสมโมเดลด้วย Nelder-Mead Optimization และค้นหาจุดตัด Threshold ที่ดีที่สุดสำหรับ F2 Metric

In [ ]:
def evaluate_f2(y_true, probs, step=0.001):
    best_f2 = 0
    best_t = 0.5
    thresholds = np.arange(0.15, 0.45, step)
    for t in thresholds:
        score = fbeta_score(y_true, (probs >= t).astype(int), beta=2, zero_division=0)
        if score > best_f2:
            best_f2 = score
            best_t = t
    prec = precision_score(y_true, (probs >= best_t).astype(int), zero_division=0)
    rec = recall_score(y_true, (probs >= best_t).astype(int), zero_division=0)
    auc = roc_auc_score(y_true, probs)
    return best_f2, best_t, prec, rec, auc

def objective(weights):
    w = np.exp(weights) / np.sum(np.exp(weights))
    blend = w[0] * oof_lgb_p + w[1] * oof_xgb + w[2] * oof_cat
    f2, _, _, _, _ = evaluate_f2(y_labeled, blend, step=0.005)
    return -f2

res = minimize(objective, [0.5, 0.3, 0.2], method='Nelder-Mead')
opt_w = np.exp(res.x) / np.sum(np.exp(res.x))
print(f"น้ำหนักการรวมโมเดล: LGB_Pseudo={opt_w[0]:.4f}, XGB={opt_w[1]:.4f}, Cat={opt_w[2]:.4f}")

oof_final = opt_w[0] * oof_lgb_p + opt_w[1] * oof_xgb + opt_w[2] * oof_cat
test_final = opt_w[0] * test_lgb_p + opt_w[1] * test_xgb + opt_w[2] * test_cat

best_f2, best_t, best_p, best_r, auc_final = evaluate_f2(y_labeled, oof_final, step=0.001)
print(f"\n=== ผลการประเมิน Out-of-Fold สูงสุด ===")
print(f"OOF F2 Score:      {best_f2:.5f}")
print(f"ROC-AUC:           {auc_final:.5f}")
print(f"Optimal Threshold: {best_t:.4f}")
print(f"Recall:            {best_r:.4f}")
print(f"Precision:         {best_p:.4f}")

## 6. Export Final Predictions
ส่งออกผลลัพธ์ลงไฟล์ `submissions/sub_task1.csv` เพื่อนำไปรวมกับโมเดลของเพื่อนร่วมทีม

In [ ]:
test_pred_binary = (test_final >= best_t).astype(int)

sub_df = pd.DataFrame({
    "id": sample_sub["id"],
    "task1": test_pred_binary
})

assert len(sub_df) == 25889
assert set(sub_df["task1"].unique()).issubset({0, 1})
assert sub_df["task1"].isna().sum() == 0

os.makedirs("../submissions", exist_ok=True)
sub_df.to_csv("../submissions/sub_task1.csv", index=False)
print(f"บันทึกไฟล์สำเร็จ: submissions/sub_task1.csv")
print(f"- ทำนายเป็นคนปกติ (0): {(test_pred_binary == 0).sum():,} คน ({(test_pred_binary == 0).mean()*100:.2f}%)")
print(f"- ทำนายเป็นคนโกง (1):   {(test_pred_binary == 1).sum():,} คน ({(test_pred_binary == 1).mean()*100:.2f}%)")
sub_df.head(10)